In [6]:
import numpy as np
import copy
from IPython.display import display, Audio
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from config import SR, MEMBRANE, HAMMER, SYSTEM
from synthesizer import TonbakSynthesizer

def sweep_parameter(target_dict_name, param_name, values_list, hit_r=0.03, duration=1.0):
    print(f"--- Sweeping {target_dict_name} -> {param_name} ---")
    
    # Create fresh copies of the base config
    mem_cfg = copy.deepcopy(MEMBRANE)
    ham_cfg = copy.deepcopy(HAMMER)
    sys_cfg = copy.deepcopy(SYSTEM)
    
    # Map string names to the actual dictionaries
    config_map = {
        'MEMBRANE': mem_cfg,
        'HAMMER': ham_cfg,
        'SYSTEM': sys_cfg
    }
    
    for val in values_list:
        if target_dict_name in config_map:
            config_map[target_dict_name][param_name] = val
            
            # Recalculate dependent variables if membrane thickness/modulus changes
            mem_cfg['surface_density'] = mem_cfg['density'] * mem_cfg['thickness']
            mem_cfg['bending_stiffness'] = (mem_cfg['young_modulus'] * mem_cfg['thickness']**3) / (12 * (1 - mem_cfg['poisson_ratio']**2))
            
        current_hit_r = val if target_dict_name == 'LOCATION' else hit_r
        
        synth = TonbakSynthesizer(
            hit_r=current_hit_r, hit_phi=0.0, 
            pickup_r=0.08, pickup_phi=0.0,
            membrane_cfg=mem_cfg, hammer_cfg=ham_cfg, system_cfg=sys_cfg
        )
        
        wave = synth.generate(duration_sec=duration)
        
        # Print the current value and display an independent audio player
        print(f"{param_name} = {val}")
        display(Audio(wave, rate=SR))

In [7]:
print("Baseline Tonbak Strike")
synth_base = TonbakSynthesizer(
    hit_r=0.03, hit_phi=0.0, 
    pickup_r=0.08, pickup_phi=0.0,
    membrane_cfg=MEMBRANE, hammer_cfg=HAMMER, system_cfg=SYSTEM
)
base_wave = synth_base.generate(duration_sec=1.5)
display(Audio(base_wave, rate=SR))

Baseline Tonbak Strike


In [8]:
# Sweeping hit_r from dead center (0.01) to the rim (0.12)
sweep_parameter('LOCATION', 'hit_r', [0.01, 0.04, 0.07, 0.10, 0.12])

--- Sweeping LOCATION -> hit_r ---
hit_r = 0.01


hit_r = 0.04


hit_r = 0.07


hit_r = 0.1


hit_r = 0.12


In [9]:
# Sweeping d3 from metallic (2e-5) to heavily damped cardboard (5e-3)
sweep_parameter('MEMBRANE', 'd3', [2e-5, 2e-4, 8e-4, 2e-3, 5e-3])

--- Sweeping MEMBRANE -> d3 ---
d3 = 2e-05


d3 = 0.0002


d3 = 0.0008


d3 = 0.002


d3 = 0.005


In [10]:
# Sweeping stiffness from soft flesh (1e4) to hard bone/wood (2e5)
sweep_parameter('HAMMER', 'stiffness', [10000, 40000, 80000, 150000, 200000])

--- Sweeping HAMMER -> stiffness ---
stiffness = 10000


stiffness = 40000


stiffness = 80000


stiffness = 150000


stiffness = 200000


In [11]:
# Sweeping base tension from loose bass (1000) to tight treble (4000)
sweep_parameter('MEMBRANE', 'tension_0', [1000, 1500, 2000, 3000, 4000])

--- Sweeping MEMBRANE -> tension_0 ---
tension_0 = 1000


tension_0 = 1500


tension_0 = 2000


tension_0 = 3000


tension_0 = 4000


In [12]:
# Sweeping alpha from flat (1.0) to sharp/pointed (2.5)
sweep_parameter('HAMMER', 'alpha', [1.0, 1.2, 1.5, 2.0, 2.5])

--- Sweeping HAMMER -> alpha ---
alpha = 1.0


alpha = 1.2


alpha = 1.5


alpha = 2.0


alpha = 2.5


In [13]:
# Sweeping air stiffness from an open frame (0) to a highly pressurized goblet (2000)
sweep_parameter('SYSTEM', 'air_stiffness', [0.0, 200.0, 800.0, 1500.0, 3000.0])

--- Sweeping SYSTEM -> air_stiffness ---
air_stiffness = 0.0


air_stiffness = 200.0


air_stiffness = 800.0


air_stiffness = 1500.0


air_stiffness = 3000.0
